In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
#Verify the excel file exist
dbutils.fs.ls("/Volumes/workspace/default/mydatabrick/")

In [0]:
#Read Excel File into Pandas
import pandas as pd

excel_path = "/Volumes/workspace/default/mydatabrick/Kenyas_Agricultural_Production.xlsx"

pdf = pd.read_excel(excel_path)

pdf.head()

In [0]:
#Check columns in the dataset
pdf.columns

In [0]:
#Convert Pandas DataFrame to Spark DataFrame
# Explicitly convert object columns to string to avoid Arrow conversion issues
for col in pdf.select_dtypes(include=['object']).columns:
    pdf[col] = pdf[col].astype(str)

df = spark.createDataFrame(pdf)

df.show()

df.printSchema()

In [0]:
#Data Cleaning and Transformations
from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
#Removing duplicate records
df_clean = df.dropDuplicates()

In [0]:
#Remove missing records
df_clean = df_clean.dropna()

In [0]:
#Cleanng of column names
df_clean = df_clean.withColumnRenamed(
    "Production Tons",
    "Production"
)

In [0]:
#Check your actual column names
df_clean.columns

In [0]:
#Convert numeric columns
df_clean = df_clean.withColumn(
    "Year",
    col("Year").cast("integer")
)


df_clean = df_clean.withColumn(
    "Production",
    col("Production").cast("double")
)

In [0]:
#Creating a new analysis column
df_clean = df_clean.withColumn(
    "Yield_Per_Hectare",
    col("Production") / col("Area_Hectares")
)

In [0]:
df_clean.show()

In [0]:
#Registering Spark SQL Table
df_clean.createOrReplaceTempView(
    "agriculture_production"
)

In [0]:
%sql
SELECT *
FROM agriculture_production
WHERE Area='Central';

In [0]:
spark.sql("""
SELECT *
FROM agriculture_production
WHERE Area='Central'
""").show()

In [0]:
#Production by Harvest Year
spark.sql("""
SELECT
    Year,
    SUM(Value) AS Total_Production
FROM agriculture_production
GROUP BY Year
ORDER BY Year
""").show()

In [0]:
#Top Producing Crops
spark.sql("""
SELECT
    Item,
    SUM(Value) AS Total_Production
FROM agriculture_production
GROUP BY Item
ORDER BY Total_Production DESC
LIMIT 10
""").show()

In [0]:
#Regional Productivity
spark.sql("""
SELECT
    Area,
    AVG(Value) AS Average_Value
FROM agriculture_production
GROUP BY Area
ORDER BY Average_Value DESC
""").show()

In [0]:
# Total Proiduction
spark.sql("""
SELECT
    Area,
    SUM(Value) AS Total_Production
FROM agriculture_production
GROUP BY Area
ORDER BY Total_Production DESC
LIMIT 10
""").show()